# Built-in DSPy and HuggingFace Datasets

Load HotPotQA and GSM8K through DSPy's built-in dataset wrappers, then pull SQuAD and TweetEval from the HuggingFace Hub and convert each one into `dspy.Example` objects you can feed into an optimizer.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

In [ ]:
%pip install -r ../requirements.txt -q
%pip install datasets -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## HotPotQA — multi-hop question answering

Requires reasoning over multiple Wikipedia paragraphs to answer a question.

In [ ]:
from dspy.datasets import HotPotQA

# Load dataset
dataset = HotPotQA(train_seed=1, train_size=100, eval_seed=2024, dev_size=50, test_size=0)

# Convert to DSPy format
train_set = [dspy.Example(question=x.question, answer=x.answer).with_inputs('question')
            for x in dataset.train]

# Example
print(train_set[0].question)
print(train_set[0].answer)

## GSM8K — grade-school math word problems

Tests multi-step arithmetic reasoning.

In [ ]:
from dspy.datasets import GSM8K

dataset = GSM8K()
train_set = [dspy.Example(question=x.question, answer=x.answer).with_inputs('question')
            for x in dataset.train[:100]]

print(train_set[0].question)
print(train_set[0].answer)

## SQuAD via HuggingFace

`datasets.load_dataset` pulls the Stanford Question Answering Dataset. Convert each row into a `dspy.Example` with the question, context, and the first listed answer.

In [ ]:
from datasets import load_dataset

# Load from HuggingFace
hf_dataset = load_dataset("squad")

# Convert to DSPy format
def convert_to_dspy(hf_example):
    return dspy.Example(
        question=hf_example['question'],
        context=hf_example['context'],
        answer=hf_example['answers']['text'][0],  # SQuAD has multiple answers, take first
    ).with_inputs('question', 'context')

train_set = [convert_to_dspy(ex) for ex in hf_dataset['train'].select(range(1000))]

print(train_set[0].question)
print(train_set[0].answer)

## TweetEval sentiment — a classification dataset

Map the integer labels back to readable strings before wrapping each row as a `dspy.Example`.

In [ ]:
# Twitter sentiment analysis
hf_dataset = load_dataset("tweet_eval", "sentiment")

def convert_to_dspy(hf_example):
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    return dspy.Example(
        text=hf_example['text'],
        sentiment=label_map[hf_example['label']]
    ).with_inputs('text')

train_set = [convert_to_dspy(ex) for ex in hf_dataset['train']]

print(f"Loaded {len(train_set)} sentiment examples")
print(train_set[0].text, '->', train_set[0].sentiment)